# 06 - Y-chromosome lineages in the KZ-219 cohort

![cohort](https://img.shields.io/badge/cohort-KZ--219-1b9e77)
![platform](https://img.shields.io/badge/platform-DNBSEQ--G400-7570b3)
![pipeline](https://img.shields.io/badge/DRAGEN-v3.10.16-d95f02)
![reference](https://img.shields.io/badge/reference-GRCh37%2Fhg19-e7298a)
![annotation](https://img.shields.io/badge/ANNOVAR-v2025Mar02-666666)
![bioproject](https://img.shields.io/badge/BioProject-PRJNA1224597-333333)

| | |
|---|---|
| **Cohort** | KZ-219 - 219 healthy Kazakh adults |
| **Sequencing** | DNBSEQ-G400, PCR-free, 100 bp paired-end |
| **Alignment and variant calling** | Illumina DRAGEN Bio-IT Platform v3.10.16 |
| **Reference** | GRCh37 / hg19 (DRAGEN hash table v8) |
| **Annotation** | ANNOVAR v2025Mar02 |
| **Y calling** | Y-LineageTracker |
| **Raw data** | NCBI SRA, BioProject PRJNA1224597 |

### Manuscript items produced here

| Item | File |
|---|---|
| Fig. 7 | `Fig7_chrY_composition.tiff` / `.png` |
| Supplementary Tables 27-30 | `Supplementary_Tables_06.docx` |
| Supplementary Data 6 | `Supplementary_Data_6_chrY.xlsx` |

### Running

| Variable | Default | Content |
|---|---|---|
| `KZ_CHRY_DIR` | `data/chrY` | `kz_219_ylt.hapresult.hg` and `kz_219_ylt.lineageresult.txt` (Y-LineageTracker v1.3.0, `classify -b 37`), `metada_with_yhaplogroups.csv` (paternal tribe and zhuz) |
| `KZ_OUT_DIR` | `results/06_chrY` | figure, Supplementary Data 6, Supplementary Tables, and `kz_219_chrY_lineage_long.tsv` (long-format calls for the paired mtDNA and chrY comparison) |

```bash
export KZ_CHRY_DIR=data/chrY
pip install -r requirements.txt
jupyter nbconvert --to notebook --execute --inplace 06_chrY_analysis.ipynb
```

Upstream, chrY calls of the 74 XY genomes from the DRAGEN v3.10.16 / GRCh37 pipeline were
classified with Y-LineageTracker against the ISOGG 2019 tree distributed with that release
(Methods). The metadata file holds self-reported tribal affiliation and is not part of the
repository.

Figures are written as TIFF (CMYK, LZW) and PNG (RGB) at 1200 dpi, Arial 8 pt with 10 pt
panel letters.

## 1 - Setup

In [ ]:
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

import scipy
import openpyxl
print('pandas', pd.__version__, '| numpy', np.__version__,
      '| scipy', scipy.__version__, '| matplotlib', plt.matplotlib.__version__,
      '| openpyxl', openpyxl.__version__)

In [ ]:
MM = 1 / 25.4
W1, W15, W2 = 88 * MM, 120 * MM, 180 * MM

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans'],
    'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 8,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'legend.title_fontsize': 8, 'hatch.linewidth': 0.5,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'xtick.major.size': 2, 'ytick.major.size': 2,
    'axes.edgecolor': '#3C3C3C', 'xtick.color': '#3C3C3C', 'ytick.color': '#3C3C3C',
    'text.color': '#1A1A1A', 'axes.labelcolor': '#1A1A1A',
    'grid.linewidth': 0.4, 'grid.color': '#C8C8C8',
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    'figure.facecolor': 'white', 'savefig.facecolor': 'white',
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.02,
})

INK, RULE, FAINT, BAND = '#1A1A1A', '#3C3C3C', '#B0B0B0', '#4A4A4A'
FS, FS_PANEL, FS_MAX = 8, 10, 12
FIG_FORMATS = ('tiff', 'png')      
ANC_HATCH = {'East': None, 'West': '////', 'Unassigned': '....'}

MAIN_FIG = {'chrY_composition': 7}
SUPP_NUM = {'tab_haplogroups': 27, 'tab_composition': 28, 'tab_diversity': 29,
            'tab_tribe_by_major': 30, 'data': 6}

ANC_ORDER  = ['East', 'West', 'Unassigned']
ANC_COLORS = {'East': '#2C6E9B', 'West': '#B5502F', 'Unassigned': '#A0A0A0'}
ANC_LABEL  = {'East': 'East Eurasian', 'West': 'West Eurasian',
              'Unassigned': 'Unresolved'}
HEAT = mcolors.LinearSegmentedColormap.from_list(
    'kz_slate', ['#FFFFFF', '#DCE4EA', '#AEBECB', '#6E8598', '#3A4E60', '#1B2A36'],
    N=256)


def on(hex_color):
    r, g, b = mcolors.to_rgb(hex_color)
    return '#FFFFFF' if 0.299 * r + 0.587 * g + 0.114 * b < 0.55 else INK


def clade_palette(clades, anc_of):
    ramps = {'East': ('#12405F', '#C3D8E5'), 'West': ('#7A3418', '#EDCBB8'),
             'Unassigned': ('#8A8A8A', '#D8D8D8')}
    out = {}
    for a in ANC_ORDER:
        sub = [c for c in clades if anc_of.get(c) == a]
        if not sub:
            continue
        cm = mcolors.LinearSegmentedColormap.from_list('ramp', list(ramps[a]))
        for i, c in enumerate(sub):
            out[c] = mcolors.to_hex(cm(i / max(1, len(sub) - 1)))
    return out


def panel(ax, letter, dx=-0.06, dy=1.02):
    ax.text(dx, dy, letter, transform=ax.transAxes, fontsize=FS_PANEL, fontweight='bold',
            va='bottom', ha='left', color=INK)


def despine(ax, keep=('left', 'bottom')):
    for name, sp in ax.spines.items():
        sp.set_visible(name in keep)


def _enforce_text(fig):
    """Raise any text below 8 pt (cap above 12 pt) and report what changed."""
    changed = []
    for t in fig.findobj(plt.matplotlib.text.Text):
        if not t.get_visible() or not t.get_text().strip():
            continue
        s = t.get_fontsize()
        if s < FS - 1e-6 or s > FS_MAX + 1e-6:
            changed.append((t.get_text()[:24], round(s, 2)))
            t.set_fontsize(min(max(s, FS), FS_MAX))
    if changed:
        print(f'  NOTE: {len(changed)} text item(s) raised to {FS} pt, check the layout: '
              f'{changed[:6]}')
    return changed


def save_fig(fig, stem, outdir='.', dpi=1200, formats=FIG_FORMATS):
    """PNG (RGB) and TIFF (CMYK, LZW) at 1200 dpi, named with the figure number."""
    from PIL import Image
    _enforce_text(fig)
    png = os.path.join(outdir, f'{stem}.png')
    fig.savefig(png, dpi=dpi, bbox_inches='tight', pad_inches=0.03)
    paths = []
    if 'tiff' in formats:
        tif = os.path.join(outdir, f'{stem}.tiff')
        with Image.open(png) as im:
            im.convert('RGB').convert('CMYK').save(tif, compression='tiff_lzw', dpi=(dpi, dpi))
        paths.append(tif)
    if 'png' in formats:
        paths.append(png)
    else:
        os.remove(png)
    for p in paths:
        with Image.open(p) as im:
            w, h, mode = *im.size, im.mode
        wmm = w / dpi * 25.4
        print(f'saved {os.path.basename(p)}: {mode}, {wmm:.0f} x {h / dpi * 25.4:.0f} mm '
              f'at {dpi} dpi, {os.path.getsize(p) / 1e6:.1f} MB'
              + ('   WARNING: wider than 183 mm' if wmm > 183 else ''))
    return paths


def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z / d * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return p, max(0.0, c - h), min(1.0, c + h)

## 2 - Configuration

In [ ]:
WORK_DIR = os.environ.get('KZ_CHRY_DIR', 'data/chrY')       # Y-LineageTracker output and metadata
OUT_DIR  = os.environ.get('KZ_OUT_DIR', 'results/06_chrY')

HAP_FILE  = 'kz_219_ylt.hapresult.hg'           # Y-LineageTracker --hapresult
LIN_FILE  = 'kz_219_ylt.lineageresult.txt'      # Y-LineageTracker --lineageresult
META_FILE = 'metada_with_yhaplogroups.csv'      # paternal tribe / zhuz

SUPPORT_THRESHOLD = 0.50 
MIN_TRIBE_N       = 10    
DPI               = 1200

XLSX_OUT = f"Supplementary_Data_{SUPP_NUM['data']}_chrY.xlsx"
LONG_OUT = 'kz_219_chrY_lineage_long.tsv'

assert os.path.isdir(WORK_DIR), f'input folder not found: {WORK_DIR} (set KZ_CHRY_DIR)'
os.makedirs(OUT_DIR, exist_ok=True)
print('WORK_DIR:', WORK_DIR)
print('OUT_DIR :', OUT_DIR)

In [ ]:
def find_file(name, extra_dirs=()):
    if os.path.isabs(name) and os.path.exists(name):
        return name
    for d in [WORK_DIR, '.', OUT_DIR, 'data', *extra_dirs]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'{name!r} not found; check the Configuration cell.')


def read_table(path, **kw):
    path = find_file(path)
    with open(path, encoding='utf-8-sig', errors='replace') as fh:
        head = fh.readline()
    sep = max([',', ';', '\t', '|'], key=head.count)
    df = pd.read_csv(path, sep=sep, encoding='utf-8-sig', **kw)
    df.columns = [str(c).strip() for c in df.columns]
    return df


def pick_col(df, *names):
    low = {str(c).lower().strip(): c for c in df.columns}
    for n in names:
        if n in low:
            return low[n]
    return None


def rename_unique(df, mapping):
    mapping = {k: v for k, v in mapping.items() if k is not None}
    out = df.copy()
    clash = [v for k, v in mapping.items() if v in out.columns and v != k]
    out = out.drop(columns=clash).rename(columns=mapping)
    return out.loc[:, ~out.columns.duplicated()]

In [ ]:
def blocks(h):
    s = str(h).strip().rstrip('~*!')
    s = s.split('+')[0].split('(')[0].strip()
    return re.findall(r'[A-Za-z]+|\d+', s)


def collapse_hg(h, level=None):
    b = blocks(h)
    if not b:
        return str(h)
    return ''.join(b[:level]) if level else ''.join(b)


Y_MAJOR_ORDER = ['C2a', 'C2b', 'C2', 'C1', 'C', 'D1', 'D', 'O1', 'O2', 'O3', 'O',
                 'N1a', 'N1b', 'N1', 'N', 'Q1a', 'Q1b', 'Q1', 'Q',
                 'R1a', 'R1b', 'R1', 'R2', 'R', 'G1', 'G2', 'G', 'J1', 'J2', 'J',
                 'L1', 'L', 'T1', 'T', 'E1b', 'E', 'I1', 'I2', 'I', 'H']
_Y_PREFIX = sorted(set(Y_MAJOR_ORDER), key=len, reverse=True)


def get_major_y(h):
    # Longest known prefix: 'C2a1a2a2a' -> 'C2a', 'R1b1a1a1a' -> 'R1b'.
    s = collapse_hg(h)
    for p in _Y_PREFIX:
        if s.startswith(p):
            return p
    return s[:2] if s else '?'

Y_GROUP = {**{c: 'EEA' for c in ['C2a', 'C2b', 'C2', 'C1', 'C', 'D1', 'D',
                                 'O1', 'O2', 'O3', 'O']},
           **{c: 'NEA' for c in ['N1a', 'N1b', 'N1', 'N', 'Q1a', 'Q1b', 'Q1', 'Q']},
           **{c: 'STP' for c in ['R1a', 'R1b', 'R1']},
           **{c: 'WSA' for c in ['R2', 'R', 'G1', 'G2', 'G', 'J1', 'J2', 'J',
                                 'L1', 'L', 'T1', 'T', 'E1b', 'E', 'I1', 'I2',
                                 'I', 'H']}}
Y_GROUP_LABEL = {'EEA': 'East Eurasian (C, O)', 'NEA': 'North Eurasian (N, Q)',
                 'STP': 'Steppe (R1)', 'WSA': 'West / South Asian'}
Y_GROUP_ORDER = ['EEA', 'NEA', 'STP', 'WSA']

_Y_LETTER = {'C': 'East', 'D': 'East', 'O': 'East', 'N': 'East', 'Q': 'East',
             'G': 'West', 'J': 'West', 'L': 'West', 'T': 'West', 'E': 'West',
             'I': 'West', 'H': 'West'}


def y_ancestry(h):
    b = blocks(h)
    if not b:
        return 'Unassigned'
    return 'West' if b[0] == 'R' else _Y_LETTER.get(b[0], 'Unassigned')


def ancestry_of(df, key):
    a = df.groupby(key, observed=True)['Ancestry'].agg(set)
    mixed = a[a.map(len) > 1]
    if len(mixed):
        raise ValueError(f'{key} groups carrying more than one origin: '
                         f'{ {k: sorted(v) for k, v in mixed.items()} }')
    return a.map(lambda s: next(iter(s)))


_NULLS = {'', 'nan', 'none', 'na', 'n/a', '?', 'unknown', 'no', '-'}
_TRIBE_ALIAS = {'kirgyz': 'Qyrgyz', 'kirgiz': 'Qyrgyz', 'qyrgyz': 'Qyrgyz',
                'qozha': 'Qoja', 'kozha': 'Qoja', 'khoja': 'Qoja', 'qoja': 'Qoja',
                'kerei': 'Kerey', 'kerey': 'Kerey', 'kypchak': 'Kypshak',
                'qypshaq': 'Kypshak', 'kypshak': 'Kypshak', 'nayman': 'Naiman',
                'tore': 'Tore', 'nogai': 'Nogai'}
_ZHUZ_ALIAS = {'older': 'Uly', 'senior': 'Uly', 'elder': 'Uly', 'uly': 'Uly',
               'middle': 'Orta', 'midlle': 'Orta', 'orta': 'Orta',
               'junior': 'Kishi', 'younger': 'Kishi', 'kishi': 'Kishi',
               'qoja': 'Qoja', 'kozha': 'Qoja', 'tore': 'Tore',
               'qyrgyz': 'Qyrgyz', 'nogai': 'Nogai'}


def norm_tribe(t):
    t = str(t).strip().rstrip('?').strip()
    if t.lower() in _NULLS:
        return 'NA'
    t = re.split(r'\s*(?:->|\u2192|,|/)\s*', t)[0].strip().rstrip('?').strip()
    return 'NA' if t.lower() in _NULLS else _TRIBE_ALIAS.get(t.lower(), t.capitalize())


def norm_zhuz(z, tribe='NA'):
    special = {'Tore': 'Tore', 'Qoja': 'Qoja', 'Qyrgyz': 'Qyrgyz', 'Nogai': 'Nogai'}
    if tribe in special:
        return special[tribe]
    z = str(z).strip().lower()
    if z in _NULLS:
        return 'NA'
    return 'Other' if z == 'other' else _ZHUZ_ALIAS.get(z, 'NA')

## 3 - Load and harmonise

Y-LineageTracker calls are merged with the cohort metadata; tribe and zhuz labels are mapped
to the vocabulary shared with the mtDNA notebook. Terminal-node support is parsed from the
lineage track, whose leading entry has the form `C2a1a2a2a(1/4)`: markers observed over
markers defining that node.

In [ ]:
hap = read_table(HAP_FILE)
lin = read_table(LIN_FILE)
y = hap.merge(lin.drop(columns=[c for c in ('Haplogroup',) if c in lin.columns]),
              on='SampleID', how='left')
y['SampleID'] = y['SampleID'].astype(str).str.strip()

meta = read_table(META_FILE)
keep = {pick_col(meta, 'sampleid', 'sample_id', 'sample'): 'SampleID',
        pick_col(meta, 'tribe', 'father_tribe', 'paternal_tribe'): 'tribe_raw',
        pick_col(meta, 'zhuz', 'father_zhuz', 'paternal_zhuz'): 'zhuz_raw'}
for src, dst in [(('gender', 'sex'), 'sex'), (('bgi_id', 'bgi id'), 'bgi_id')]:
    col = pick_col(meta, *src)
    if col:
        keep[col] = dst
m = rename_unique(meta[list(keep)], keep)
m['SampleID'] = m['SampleID'].astype(str).str.strip()
y = y.merge(m, on='SampleID', how='left')

y['Tribe'] = y['tribe_raw'].apply(norm_tribe)
y['Zhuz'] = [norm_zhuz(z, t) for z, t in zip(y['zhuz_raw'], y['Tribe'])]

y['Terminal'] = y['Haplogroup'].apply(collapse_hg)
y['Major'] = y['Terminal'].apply(get_major_y)
y['Group'] = y['Major'].map(Y_GROUP)
y['Ancestry'] = y['Terminal'].apply(y_ancestry)
y['Provisional'] = y['Haplogroup'].astype(str).str.contains('~')


def terminal_support(track):
    first = str(track).split('->')[0].strip()
    mm = re.match(r'^([^()]+)(?:\((\d+)/(\d+)\))?$', first)
    if not mm or not mm.group(3):
        return pd.Series({'TermSupport': np.nan, 'TermMarkers': np.nan})
    return pd.Series({'TermSupport': float(mm.group(2)),
                      'TermMarkers': float(mm.group(3))})


if 'LineageTrack' in y.columns:
    y = pd.concat([y, y['LineageTrack'].apply(terminal_support)], axis=1)
    y['TreeDepth'] = y['LineageTrack'].astype(str).str.count(r'->') + 1
    y['SupportFrac'] = (y['TermSupport'] / y['TermMarkers']).round(3)
if 'Mutations' in y.columns:
    y['nMarkers'] = y['Mutations'].astype(str).apply(
        lambda s: 0 if s.strip() in ('nan', '', '.') else len(re.split(r',\s*', s.strip())))

missing = sorted(set(y['Major']) - set(Y_MAJOR_ORDER))
assert not missing, f'major clades outside Y_MAJOR_ORDER: {missing}'

LONG_COLS = ['SampleID', 'Haplogroup', 'Terminal', 'Major', 'Group', 'Ancestry',
             'Tribe', 'Zhuz']
y_long = y[LONG_COLS].copy()
y_long['Source'] = 'chrY'
y_long.to_csv(os.path.join(OUT_DIR, LONG_OUT), sep='\t', index=False)

N = len(y)
print(f'chrY calls         : {N}')
print(f'metadata matched   : {y["tribe_raw"].notna().sum()} / {N}')
print(f'terminal labels    : {y["Terminal"].nunique()}')
print(f'major clades       : {y["Major"].nunique()}')
print(f'provisional calls  : {int(y["Provisional"].sum())}')
print(f'paternal tribes    : {(y["Tribe"] != "NA").sum()} annotated, '
      f'{(y["Tribe"] == "NA").sum()} not recorded')

In [ ]:
tab = (y.groupby('Ancestry', observed=True)
        .agg(n=('SampleID', 'size'), clades=('Major', 'nunique'))
        .reindex(ANC_ORDER).dropna(how='all'))
tab['pct'] = (tab['n'] / N * 100).round(1)
print('Phylogeographic origin of paternal lineages')
print(tab.to_string())

print('\nPhylogeographic group')
for g in Y_GROUP_ORDER:
    k = int((y['Group'] == g).sum())
    if k:
        print(f'  {Y_GROUP_LABEL[g]:<26} {k:3d} ({k / N * 100:4.1f}%)')

print('\nPaternal tribe')
tr = y['Tribe'].value_counts()
for k, v in tr.items():
    print(f'  {k:<12} {v:3d} ({v / N * 100:4.1f}%)')
print(f'\nTribes with n >= {MIN_TRIBE_N}: '
      f'{[t for t in tr.index if t != "NA" and tr[t] >= MIN_TRIBE_N]}')

## 4 - Fig. 7a, b - haplogroup spectrum and composition

Bars are coloured by the phylogeographic origin of the haplogroup, on the same two-way
scheme as the mtDNA figure; the band above resolves it into the four groups conventionally
distinguished on the Y chromosome. Panel b pools tribes below `MIN_TRIBE_D`.

West Eurasian bars and segments are hatched as well as coloured. Group names in the band
above a are shortened (C, O / N, Q / R1 / G, J, L, R2).

In [ ]:
MIN_TRIBE_D = 3                 
H_A, H_ROW, GAP, TOP_IN, BOT_IN = 1.45, 0.34, 0.62, 0.30, 0.42

freq = y['Major'].value_counts()
freq = freq.reindex([c for c in Y_MAJOR_ORDER if c in freq.index])
anc_of = ancestry_of(y, 'Major')

tn = y['Tribe'].value_counts()
BIG = [t for t in tn.index if t != 'NA' and tn[t] >= MIN_TRIBE_D]
OTHER, NOREC = f'Other tribes, each n < {MIN_TRIBE_D}', 'Tribe not recorded'
groups = ([('Whole cohort', y)] + [(t, y[y['Tribe'] == t]) for t in BIG] +
          [(OTHER, y[(~y['Tribe'].isin(BIG)) & (y['Tribe'] != 'NA')]),
           (NOREC, y[y['Tribe'] == 'NA'])])
groups = [(nm, s) for nm, s in groups if len(s)]
print(f'  panel b rows: {len(groups)}  (tribes n >= {MIN_TRIBE_D}: {len(BIG)})')

H_B   = H_ROW * len(groups)
FIG_H = TOP_IN + H_A + GAP + H_B + BOT_IN
fig = plt.figure(figsize=(W2, FIG_H))
gs = fig.add_gridspec(2, 1, height_ratios=[H_A, H_B],
                      hspace=GAP / ((H_A + H_B) / 2),
                      left=0.165, right=0.855,
                      top=1 - TOP_IN / FIG_H, bottom=BOT_IN / FIG_H)

gsa = gs[0].subgridspec(2, 1, height_ratios=[0.09, 1.0], hspace=0.05)
Y_GROUP_SHORT = {'EEA': 'C, O', 'NEA': 'N, Q', 'STP': 'R1', 'WSA': 'G, J, L, R2'}
AX_W_MM = W2 / MM * (0.855 - 0.165)
axb, ax = fig.add_subplot(gsa[0]), fig.add_subplot(gsa[1])

nc = len(freq)
grp = [Y_GROUP[c] for c in freq.index]
prev, start, cuts = None, 0, []
for j, g in enumerate(grp + [None]):
    if g != prev:
        if prev is not None:
            axb.fill_betweenx([0, 1], start - .5, j - .5, color=BAND, lw=0)
            span_mm = (j - start) * AX_W_MM / nc
            lab = Y_GROUP_LABEL[prev] if len(Y_GROUP_LABEL[prev]) * 1.75 < span_mm \
                else Y_GROUP_SHORT[prev]
            axb.text((start + j - 1) / 2, .5, lab, ha='center', va='center',
                     fontsize=FS, fontweight='bold', color='white')
            if start:
                cuts.append(start)
        start = j
    prev = g
for j in cuts:
    axb.axvline(j - .5, color='white', lw=1.1)
axb.set(xlim=(-.5, nc - .5), ylim=(0, 1), xticks=[], yticks=[])
axb.spines[:].set_visible(False)

ax.bar(range(nc), freq.values, width=.7, lw=.4, edgecolor='white', zorder=3,
       color=[ANC_COLORS[anc_of[c]] for c in freq.index],
       hatch=[ANC_HATCH[anc_of[c]] for c in freq.index])
for i, v in enumerate(freq.values):
    ax.text(i, v + freq.max() * .03, f'{v}\n{v / N * 100:.0f}%', ha='center',
            va='bottom', fontsize=FS, color=INK, linespacing=1.1)
ax.set_xticks(range(nc))
ax.set_xticklabels(freq.index, fontsize=FS)
for j, t in enumerate(ax.get_xticklabels()):
    t.set_color(ANC_COLORS[anc_of[freq.index[j]]])
ax.set_xlim(-.5, nc - .5)
ax.set_ylim(0, freq.max() * 1.42)
ax.set_ylabel(f'Individuals (n = {N})')
ax.set_xlabel('Y-chromosome haplogroup', labelpad=2)
ax.grid(axis='y', zorder=0)
ax.set_axisbelow(True)
despine(ax)
for j in cuts:
    ax.axvline(j - .5, color=FAINT, lw=.5, zorder=1)
ax.legend([plt.Rectangle((0, 0), 1, 1, fc=ANC_COLORS[a], ec='white', hatch=ANC_HATCH[a])
           for a in ('East', 'West')],
          [ANC_LABEL[a] for a in ('East', 'West')], frameon=False, ncol=2,
          loc='upper left', bbox_to_anchor=(0.01, 1.00), handlelength=1.0,
          handleheight=.75, columnspacing=1.1, handletextpad=.45)
panel(axb, 'a', dx=-0.215, dy=1.5)

ax2 = fig.add_subplot(gs[1])
cols = [c for c in Y_MAJOR_ORDER if c in set(y['Major'])]
pal = clade_palette(cols, anc_of)
SEG_MIN_PCT = 2.6    
zof = y[y['Tribe'].isin(BIG)].groupby('Tribe', observed=True)['Zhuz'].agg(
    lambda s: s.mode().iat[0])

stat = [(len(s), int((s['Ancestry'] == 'East').sum()), int((s['Ancestry'] == 'West').sum()))
        for _, s in groups]
comp = np.array([[(s['Major'] == c).sum() / len(s) * 100 for c in cols] for _, s in groups])
yy = np.arange(len(groups))[::-1]
left = np.zeros(len(groups))
for j, c in enumerate(cols):
    v = comp[:, j]
    ax2.barh(yy, v, left=left, height=.58, color=pal[c], lw=.3, edgecolor='white', zorder=3,
             hatch=ANC_HATCH[anc_of[c]])
    for yv, vv, lv in zip(yy, v, left):
        if vv >= SEG_MIN_PCT * max(len(c), 1):
            ax2.text(lv + vv / 2, yv, c, ha='center', va='center', fontsize=FS,
                     color=on(pal[c]), zorder=4,
                     bbox=dict(boxstyle='square,pad=0.05', fc=pal[c], ec='none'))
    left += v

tr = ax2.get_yaxis_transform()
ax2.text(1.015, yy[0] + .58, 'East Eurasian,\n% (95% CI)', transform=tr, ha='left',
         va='bottom', fontsize=FS, color=BAND, linespacing=1.0)
for yv, (n, k, w) in zip(yy, stat):
    _, lo, hi = wilson(k, n)
    ax2.text(1.015, yv, f'{k / n * 100:.0f}  ({lo * 100:.0f}-{hi * 100:.0f})',
             transform=tr, ha='left', va='center', fontsize=FS, color=INK)

labels = []
for (name, _), (n, _, _) in zip(groups, stat):
    z = zof.get(name)
    labels.append(f'{name}' + (f' ({z})' if isinstance(z, str) and z != 'NA' else '') +
                  f'\nn = {n}')
ax2.set_yticks(yy)
ax2.set_yticklabels(labels, fontsize=FS, linespacing=1.0)
ax2.set_xlim(0, 100)
ax2.set_ylim(-.6, len(groups) - .4)
ax2.set_xticks(range(0, 101, 20))
ax2.set_xlabel('Paternal lineages (%)', labelpad=2)
ax2.tick_params(axis='y', length=0)
despine(ax2, keep=('bottom',))
panel(ax2, 'b', dx=-0.215, dy=1.03)

save_fig(fig, f"Fig{MAIN_FIG['chrY_composition']}_chrY_composition", OUT_DIR, dpi=DPI)
plt.show()

In [ ]:
top = y['Terminal'].value_counts()
print(f'Most frequent paternal lineage: {top.index[0]} '
      f'{top.iloc[0]}/{N} ({top.iloc[0] / N * 100:.1f}%)')
for t in [t for t in y['Tribe'].value_counts().index if t != 'NA'][:4]:
    sub = y[y['Tribe'] == t]
    vc = sub['Terminal'].value_counts()
    print(f'  {t:<10} n={len(sub):2d}  top {vc.index[0]} {vc.iloc[0]} '
          f'({vc.iloc[0] / len(sub) * 100:.0f}%)  distinct lineages {sub["Terminal"].nunique()}')

## 5 - Supplementary Data

The tribe by terminal-lineage matrix is tabulated rather than plotted: at ISOGG terminal
resolution most cells hold a single observation. Diversity is given as Nei's unbiased
estimator with its analytic standard deviation (Nei 1987, eq. 8.12), together with the
number of distinct lineages per individual.

In [ ]:
def nei_h_sd(labels):
    x = pd.Series(list(labels)).dropna()
    n = len(x)
    if n < 2:
        return np.nan, np.nan
    p = x.value_counts(normalize=True).values
    s2, s3 = np.sum(p ** 2), np.sum(p ** 3)
    h = n / (n - 1) * (1 - s2)
    var = 2 / (n * (n - 1)) * (2 * (n - 2) * (s3 - s2 ** 2) + s2 - s2 ** 2)
    return float(h), float(np.sqrt(var))


div_rows = []
for name, sub in [('Whole cohort', y)] + [(t, y[y['Tribe'] == t]) for t in BIG]:
    h, sd = nei_h_sd(sub['Terminal'])
    top = sub['Terminal'].value_counts()
    div_rows.append({'Marker': 'chrY', 'Group': name, 'n': len(sub),
                     'lineages': sub['Terminal'].nunique(),
                     'lineages_per_n': round(sub['Terminal'].nunique() / len(sub), 2),
                     'h': round(h, 4), 'h_SD': round(sd, 4),
                     'top_lineage': top.index[0], 'top_n': int(top.iloc[0])})

hg_summary = (y.groupby(['Major', 'Terminal', 'Ancestry'], observed=True)
                .size().rename('n').reset_index()
                .assign(group=lambda d: d['Major'].map(Y_GROUP),
                        pct=lambda d: (d['n'] / N * 100).round(2))
                .sort_values(['n', 'Major'], ascending=[False, True])
                .reset_index(drop=True))

anc_rows = []
for name, sub in groups:
    n = len(sub)
    k = int((sub['Ancestry'] == 'East').sum())
    w = int((sub['Ancestry'] == 'West').sum())
    _, lo, hi = wilson(k, n)
    anc_rows.append({'Group': name, 'n': n, 'East': k, 'West': w,
                     'Unresolved': n - k - w,
                     'East_pct': round(k / n * 100, 1),
                     'East_lo95': round(lo * 100, 1),
                     'East_hi95': round(hi * 100, 1)})

qc = y[[c for c in ['SampleID', 'bgi_id', 'Haplogroup', 'Terminal', 'Major', 'Ancestry',
                    'Provisional', 'TermSupport', 'TermMarkers', 'SupportFrac',
                    'TreeDepth', 'nMarkers'] if c in y.columns]]


def write_sheet(ws, df, key_cols=(), width_key=24, width_val=14):
    hdr_font = Font(name='Arial', bold=True, color='FFFFFF', size=10)
    hdr_fill = PatternFill('solid', start_color='FF2C6E9B')
    key_fill = PatternFill('solid', start_color='FFDCE4EA')
    alt_fill = PatternFill('solid', start_color='FFF2F5F8')
    wht_fill = PatternFill('solid', start_color='FFFFFFFF')
    ctr = Alignment(horizontal='center', vertical='center', wrap_text=True)
    lft = Alignment(horizontal='left', vertical='center', wrap_text=True)
    thin = Side(style='thin', color='BFBFBF')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    for ci, col in enumerate(df.columns, 1):
        c = ws.cell(row=1, column=ci, value=str(col))
        c.font, c.fill, c.alignment, c.border = hdr_font, hdr_fill, ctr, border
    for ri, row in enumerate(df.itertuples(index=False), 2):
        for ci, val in enumerate(row, 1):
            if isinstance(val, np.integer):
                val = int(val)
            elif isinstance(val, np.floating):
                val = None if np.isnan(val) else float(val)
            elif isinstance(val, np.bool_):
                val = bool(val)
            c = ws.cell(row=ri, column=ci, value=val)
            c.border, c.font = border, Font(name='Arial', size=10)
            if df.columns[ci - 1] in key_cols:
                c.fill, c.alignment = key_fill, lft
            else:
                c.fill = alt_fill if ri % 2 == 0 else wht_fill
                c.alignment = ctr
    ws.row_dimensions[1].height = 28
    ws.freeze_panes = 'B2'
    ws.auto_filter.ref = ws.dimensions
    for ci, col in enumerate(df.columns, 1):
        ws.column_dimensions[get_column_letter(ci)].width = (
            width_key if col in key_cols else width_val)


In [ ]:
XLSX_OUT = f"Supplementary_Data_{SUPP_NUM['data']}_chrY.xlsx"

sheets = [
    ('01_per_sample_calls',     y[LONG_COLS], ('SampleID', 'Haplogroup', 'Terminal')),
    ('02_haplogroup_counts',
     hg_summary[['group', 'Major', 'Terminal', 'Ancestry', 'n', 'pct']],
     ('group', 'Major', 'Terminal')),
    ('03_tribe_by_major',       pd.crosstab(y['Tribe'], y['Major']).reset_index(), ('Tribe',)),
    ('04_tribe_by_terminal',    pd.crosstab(y['Tribe'], y['Terminal']).reset_index(), ('Tribe',)),
    ('05_ancestry_composition', pd.DataFrame(anc_rows), ('Group',)),
    ('06_lineage_diversity',    pd.DataFrame(div_rows), ('Marker', 'Group', 'top_lineage')),
    ('07_call_quality',         qc, ('SampleID', 'Haplogroup', 'Terminal')),
]

wb = Workbook()
for i, (name, frame, keys) in enumerate(sheets):
    ws = wb.active if i == 0 else wb.create_sheet()
    ws.title = name
    write_sheet(ws, frame.reset_index(drop=True), key_cols=keys)
wb.save(os.path.join(OUT_DIR, XLSX_OUT))
print('saved:', XLSX_OUT, f'({len(sheets)} sheets)')
print(pd.DataFrame(div_rows).to_string(index=False))

In [ ]:
from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT

SUPPL_TABLES = [
    (SUPP_NUM['tab_haplogroups'], 'Y-chromosome haplogroup composition of the 74 male participants, by '
         'phylogeographic group, major clade and terminal ISOGG label.',
     hg_summary[['group', 'Major', 'Terminal', 'Ancestry', 'n', 'pct']]),
    (SUPP_NUM['tab_composition'], 'Y-chromosome phylogeographic composition of the whole male sample and of each '
         'paternal tribal stratum, with Wilson 95% confidence intervals.',
     pd.DataFrame(anc_rows)),
    (SUPP_NUM['tab_diversity'], 'Y-chromosome lineage diversity by paternal tribal stratum, Nei unbiased gene '
         'diversity over terminal lineages with analytic standard deviation.',
     pd.DataFrame(div_rows)),
    (SUPP_NUM['tab_tribe_by_major'], 'Cross-tabulation of paternal tribe against major Y-chromosome clade.',
     pd.crosstab(y['Tribe'], y['Major']).reset_index()),
]

def _fmt(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ''
    if isinstance(v, (bool, np.bool_)):
        return 'yes' if v else 'no'
    if isinstance(v, (int, np.integer)):
        return f'{int(v):,}'
    if isinstance(v, (float, np.floating)):
        return f'{v:,.4f}'.rstrip('0').rstrip('.')
    return str(v)

def add_table(doc, df, caption, tag):
    p = doc.add_paragraph()
    r = p.add_run(f'{tag}  {caption}')
    r.bold = True
    r.font.size = Pt(9)
    r.font.name = 'Arial'
    p.paragraph_format.space_after = Pt(4)
    size = Pt(7) if len(df.columns) > 9 else Pt(8)
    t = doc.add_table(rows=1, cols=len(df.columns))
    t.style = 'Table Grid'
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    for j, col in enumerate(df.columns):
        c = t.rows[0].cells[j]
        c.text = ''
        run = c.paragraphs[0].add_run(str(col))
        run.bold = True; run.font.size = size; run.font.name = 'Arial'
        c.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
    for _, row in df.iterrows():
        cells = t.add_row().cells
        for j, v in enumerate(row):
            cells[j].text = ''
            run = cells[j].paragraphs[0].add_run(_fmt(v))
            run.font.size = size; run.font.name = 'Arial'
            cells[j].paragraphs[0].alignment = (
                WD_ALIGN_PARAGRAPH.LEFT if j == 0 else WD_ALIGN_PARAGRAPH.CENTER)
    doc.add_paragraph().paragraph_format.space_after = Pt(10)

doc = Document()
for s in doc.sections:
    s.left_margin = s.right_margin = Inches(0.6)
h = doc.add_paragraph()
hr = h.add_run('Supplementary Tables 06')
hr.bold = True
hr.font.size = Pt(12)
hr.font.name = 'Arial'

written = 0
for num, cap, df in SUPPL_TABLES:
    if df is None or df.empty:
        print(f'  Supplementary Table {num}: not available, skipped'); continue
    add_table(doc, df, cap, f'Supplementary Table {num}.')
    written += 1
    print(f'  Supplementary Table {num}: {df.shape[0]} rows x {df.shape[1]} cols')

DOCX = os.path.join(OUT_DIR, 'Supplementary_Tables_06.docx')
doc.save(DOCX)
print(f'\nsaved {os.path.basename(DOCX)}: {written} tables')

## 6 - Outputs

| File | Content |
|---|---|
| `Fig7_chrY_composition.tiff` / `.png` | Fig. 7a, b - haplogroup spectrum; composition by paternal tribe |
| `Supplementary_Data_6_chrY.xlsx` | Supplementary Data 6, sheets 01-07 |
| `Supplementary_Tables_06.docx` | Supplementary Tables 27-30 |
| `kz_219_chrY_lineage_long.tsv` | long-format calls for the paired mtDNA and chrY comparison |